# Import Library

In [92]:
import importlib
import plant
import numpy as np
importlib.reload(plant)

<module 'plant' from 'c:\\Users\\alexh\\Coding\\Drone_Simulation\\plant.py'>

# Create sample input 

In [93]:
state_vec = [
    10.5, 5.0, -2.0, #position
    0.0, 1.2, 0.1, #velocity
    0.087, 0.174, 0.785, #euler angel
    0.01, -0.02, 0.1 #angular velocity
]
mass = 10
intertia = np.diag ((3,3,3))
length = 10
print (intertia)

[[3 0 0]
 [0 3 0]
 [0 0 3]]


# Sample output

In [94]:
config = plant.DroneConfig(mass, intertia, length)
sim = plant.DronePlant(config, state_vec)
phi_dot, theta_dot, psi_dot = sim.derivatives()
print ('phi rate:', phi_dot)
print ('theta rate:', theta_dot)
print ('psi rate:', psi_dot)

phi rate: 0.027205805491459185
theta rate: -0.028613386832183188
psi rate: 0.09938467606528571


# Prove the result is correct by reversing them back to angular velocity (omega)

In [95]:
phi, theta, psi = state_vec[6: 9]
wx, wy, wz = state_vec[9:12]
sin_psi, cos_psi = np.sin (psi), np.cos (psi)
sin_theta, cos_theta = np.sin (theta), np.cos (theta)
sin_phi, cos_phi = np.sin (phi), np.cos (phi)
print ('phi:', phi)
print ('theta:', theta)
print ('psi:', psi)
print ('wx:', wx)
print ('wy:', wy)
print ('wz:', wz)
print ('sin_phi:', sin_phi)
print ('cos_phi:', cos_phi)
print ('sin_theta:', sin_theta)
print ('cos_theta:', cos_theta)
print ('sin_psi:', sin_psi)
print ('cos_psi:', cos_psi)

phi: 0.087
theta: 0.174
psi: 0.785
wx: 0.01
wy: -0.02
wz: 0.1
sin_phi: 0.0868902910275923
cos_phi: 0.9962178864711978
sin_theta: 0.17312332416475054
cos_theta: 0.9849001546502806
sin_psi: 0.706825181105366
cos_psi: 0.7073882691671998


# Mapping to Angular velocity of Body Frame

$$
\boldsymbol{\omega}_b

=

T_1(\phi)T_2(\theta)
\begin{bmatrix}
0 \\
0 \\
\dot{\psi}
\end{bmatrix}

+

T_1(\phi)
\begin{bmatrix}
0 \\
\dot{\theta} \\
0
\end{bmatrix}

+

\begin{bmatrix}
\dot{\phi} \\
0 \\
0
\end{bmatrix}.
$$

Pitch Rotational Matrix
$$
T_2(\theta) = \begin{bmatrix}
\cos(\theta) & 0 & -\sin(\theta) \\
0 & 1 & 0 \\
\sin(\theta) & 0 & \cos(\theta)
\end{bmatrix}
$$

Roll Rotational Matrix
$$
T_1(\phi) = \begin{bmatrix}
1 & 0 & 0 \\
0 & \cos(\phi) & \sin(\phi) \\
0 & -\sin(\phi) & \cos(\phi)
\end{bmatrix}
$$

In [96]:
T2 = np.matrix ([
    [cos_theta, 0, -sin_theta], 
    [0, 1, 0], 
    [sin_theta, 0, cos_theta],
])
print (T2)

[[ 0.98490015  0.         -0.17312332]
 [ 0.          1.          0.        ]
 [ 0.17312332  0.          0.98490015]]


In [97]:
T1 = np.matrix ([
    [1, 0, 0], 
    [0, cos_phi, sin_phi], 
    [0, -sin_phi, cos_phi],
])
print (T1)

[[ 1.          0.          0.        ]
 [ 0.          0.99621789  0.08689029]
 [ 0.         -0.08689029  0.99621789]]


In [98]:
psi_dot_vector = np.matrix([0, 0, psi_dot]).T
theta_dot_vector = np.matrix([0, theta_dot, 0]).T
phi_dot_vector = np.matrix([phi_dot, 0, 0]).T
print ('psi_vector: \n', psi_dot_vector); print()
print ('theta_vector: \n', theta_dot_vector); print()
print ('phi_vector: \n', phi_dot_vector); print()

psi_vector: 
 [[0.        ]
 [0.        ]
 [0.09938468]]

theta_vector: 
 [[ 0.        ]
 [-0.02861339]
 [ 0.        ]]

phi_vector: 
 [[0.02720581]
 [0.        ]
 [0.        ]]



$$
\boldsymbol{\omega}_b

=

T_1(\phi)T_2(\theta)
\begin{bmatrix}
0 \\
0 \\
\dot{\psi}
\end{bmatrix}

+

T_1(\phi)
\begin{bmatrix}
0 \\
\dot{\theta} \\
0
\end{bmatrix}

+

\begin{bmatrix}
\dot{\phi} \\
0 \\
0
\end{bmatrix}.
$$

In [99]:
wb = T1 @ T2 @ psi_dot_vector + T1 @ theta_dot_vector + phi_dot_vector

print ('Angular Velocity Matrix: \n', wb); print()
print ('Original angular velocity: ')
print ('wx: ', wx)
print ('wy: ', wy)
print ('wz: ', wz)

Angular Velocity Matrix: 
 [[ 0.01]
 [-0.02]
 [ 0.1 ]]

Original angular velocity: 
wx:  0.01
wy:  -0.02
wz:  0.1


As can be seen, the original angular velocity is the same, hence the code is implemented correctly